# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadbinijaz17/flyrankAI_Intern_ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Bootstrap: run identically in Colab and locally (copied from the starter notebooks).
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/muhammadbinijaz17/flyrankAI_Intern_ML"
REPO_DIR = "flyrankAI_Intern_ML"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # walk up to the repo root (folder containing data/raw)
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Working dir:", os.getcwd())
print("Starter data found. You're ready.")

Working dir: D:\FlyrankAI\flyrankAI_Intern_ML
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Scoring + ranking.** My lane is **Refresh / Content Opportunity Scoring**, and the decision it supports — *which page should an editor look at first?* — is a ranking decision, not a grouping problem and not a single yes/no gate.

- Underneath I train a **binary classifier** (predict probability of decline), but the classifier is only the engine. What ships is a **priority score** that orders the review queue — the same ranked artifact the starter pipeline already exposes (`refresh_queue_sample.csv`, Precision@50).
- **Why ranking/scoring, not bare classification:** the only pages that get acted on are the top of the queue. An editor reviews a small, fixed number of pages per cycle, so what I need to be right about is the *order*. With a 54.2% declining base rate, random picking is already ~54% "accurate" — plain accuracy/classification cannot tell you whether the top 50 are the right top 50.
- **Error cost is directional.** A page that should sit at #3 but ranks at #300 costs an editor a missed urgent refresh; a swap between #35 and #36 costs almost nothing. Ranking/scoring makes that directional cost the thing I optimize, which is exactly what the four framing questions ask me to minimize.

In [2]:
# Section 1 check: the review pool is huge, so order matters.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
base_rate = df['is_declining_label'].mean()

print(f"Candidate pages to review: {len(df):,}")
print(f"Share actually declining (base rate): {base_rate:.1%}")
print("Random Precision@50 equals the base rate, so a good ranking must do better.")

Candidate pages to review: 30,000
Share actually declining (base rate): 54.2%
Random Precision@50 equals the base rate, so a good ranking must do better.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What I predict:** `is_declining_label` — 1 when the page has declined, else 0. It is built as `is_declining_label = (trend_direction == "down")`, and `trend_direction == "down"` means last-30-day impressions fell by more than 20% versus the previous 30 days (`trend_pct < −20`). 16,262 of 30,000 pages (54.2%) are labeled 1.

**Honest answer — it is a defined rule on measured data, so it is a *proxy*, not a purely observed outcome:**

- No human labeled these rows. The label is an operationalization of measured search-impression movement across two 30-day windows. It is *grounded in observed data*, but the −20% threshold is defined by a rule.
- **The label trap (from the data skill):** because the label derives from `trend_pct`, the columns `trend_direction` and `trend_pct` must **never** be features — a model given them would just restate the rule (leakage), not learn about the world. The point of the model is to anticipate the drop from the *other* columns.
- **Contemporaneous-window caveat:** in the starter CSV the features and the label describe the *same* trailing 90-day window, so the label really captures current state ("this page is declining now"), not a clean past→future outcome. On the warehouse release I will fix this: features from a mid-panel month (e.g. `month=2026-03`) and the label from a **forward** month, so the target becomes a genuinely observed future outcome, with the final month sealed as the test month.

In [3]:
# Section 2 check: reproduce the −20% rule and confirm the label is rule-derived.
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

reproduced_down = df['trend_pct'].fillna(0) < -20
match = (reproduced_down == (df['trend_direction'] == 'down')).mean()
print(f'Rows where the -20% rule reproduces trend_direction==down: {match:.1%}')

print('Columns EXCLUDED from features (label source): trend_direction, trend_pct')
print(f'Label balance (declining share): {(df["trend_direction"] == "down").mean():.1%}')

Rows where the -20% rule reproduces trend_direction==down: 100.0%
Columns EXCLUDED from features (label source): trend_direction, trend_pct
Label balance (declining share): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50** — of the top 50 pages the model ranks highest, the fraction that truly need a refresh (i.e. are declining).

- **Why this one:** the only pages anyone acts on are the top of the queue. In a refresh cycle an editor reviews a small, fixed number of pages, so the defensible number is: *of the top 50 I put in front of the editor, how many were right?* Precision@50 is exactly that.
- **What "good" means (the bar to beat):**
  - Random picking: Precision@50 ≈ base rate ≈ **0.542**.
  - Naive "old pages first" rule: wrong in both directions (shown in section 5).
  - Starter pipeline: logistic regression ≈ 0.627 ROC-AUC; random forest ≈ 0.750 ROC-AUC and **0.740 Precision@50**.
  - **Good = beat 0.740 Precision@50 on a client-holdout split** (never on training clients), without collapsing recall.
- **Secondary sanity metric:** ROC-AUC for overall ranking quality across the whole queue, not just the top.
- **Why not accuracy:** with a 54.2% base rate an "everything is declining" model scores ~54% accuracy while handing the editor a useless queue. Accuracy cannot grade the top of a ranking.

In [4]:
# Section 3 check: set the bar for "good".
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
base_rate = (df['trend_direction'] == 'down').mean()

print(f'Random-chance Precision@50 (= base rate): {base_rate:.3f}')
print('Starter random forest Precision@50: 0.740')
print('Definition of good: beat 0.740 Precision@50 on a client-holdout split.')

Random-chance Precision@50 (= base rate): 0.542
Starter random forest Precision@50: 0.740
Definition of good: beat 0.740 Precision@50 on a client-holdout split.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (one page).** The lane slice is the starter release: 30,000 rows × 44 columns, one row per pseudonymized page, 32 clients, trailing-90-day metrics. Below I load it, confirm the shape, verify `content_id` is unique (safe for joins/splits), and show the head. `client_id` is the grouping key for client-holdout validation — never a feature.

In [5]:
# Section 4: the unit of analysis, as a real dataframe.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print('Shape:', df.shape)
print('Unique content_id:', df['content_id'].nunique())
print('Unique client_id:', df['client_id'].nunique())

cols = ['content_id', 'client_id', 'content_type', 'content_age_days', 'impressions_90d',
        'clicks_90d', 'ctr', 'avg_position', 'is_declining_label']
display(df[cols].head(10))
print('One row = one page. Label balance:')
print(df['is_declining_label'].value_counts(normalize=True).round(3))

Shape: (30000, 45)
Unique content_id: 30000
Unique client_id: 32
             content_id          client_id  ... avg_position  is_declining_label
0  content_304f48230142  client_f369cb89fc  ...         10.6                   1
1  content_a1fb4e703a9e  client_4e07408562  ...         20.3                   1
2  content_9aa793d4d895  client_7f2253d7e2  ...         36.5                   1
3  content_331d6c4de07b  client_19581e27de  ...          6.2                   0
4  content_d99b7a2d90ca  client_3fdba35f04  ...         44.0                   1
5  content_d4084a4bc775  client_f369cb89fc  ...          8.5                   1
6  content_9a34b442b552  client_8722616204  ...          7.0                   1
7  content_a63219c6e95a  client_19581e27de  ...         21.2                   0
8  content_5e6c160719bc  client_6208ef0f77  ...         46.0                   1
9  content_c27558df2b0c  client_19581e27de  ...          4.9                   1

[10 rows x 9 columns]
One row = one page. L

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Several reasons, all checkable in the data:

- **Signals conflict.** The obvious rule — "refresh old pages" — fails in both directions: some old pages are still performing fine, and some young pages are already dropping (checked below). Age alone cannot separate the groups, so any single-threshold if-statement mislabels thousands of pages.
- **The real pattern is interactive.** A high-impression page slipping in rank is far more urgent than a low-traffic page with the same percentage drop. That is a volume × position × trend interaction, and the weighting changes yet again for pages with no position data (`avg_position = 0`, 1,205 rows) and for pages with systematically missing keyword metadata (missingness tracks `content_type`). Hard-coding every branch is a maintenance nightmare and still fails on unseen combinations.
- **The rule that would work IS the label** — `trend_pct < −20` — and it is forbidden as a feature (leakage). The honest problem is to *anticipate* the drop from the other columns, and that mapping is noisy and many-to-many.
- **Noise.** Rate columns are ×100 percentages (`ctr = 0.76` means 0.76%), `scroll_rate`/`ai_traffic_pct` can exceed 100, and 54.2% of pages are declining. A model can average many weak, noisy signals into a stable score; a few fixed thresholds cannot.

In [6]:
# Section 5 check: a naive "refresh the old pages" rule is wrong in both directions.
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

old = df['content_age_days'] > 365
young = ~old

old_fine = ((df['is_declining_label'] == 0) & old).sum()
young_declining = ((df['is_declining_label'] == 1) & young).sum()

print(f'Old pages (age > 365d) that are NOT declining: {old_fine:,} ({old_fine / old.sum():.0%} of old pages)')
print(f'Young pages (age <= 365d) that ARE declining: {young_declining:,} ({young_declining / young.sum():.0%} of young pages)')
print('A single age cut-off is wrong in both directions — the signal is multi-feature.')

Old pages (age > 365d) that are NOT declining: 3,649 (57% of old pages)
Young pages (age <= 365d) that ARE declining: 13,551 (57% of young pages)
A single age cut-off is wrong in both directions — the signal is multi-feature.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.